In [1]:
import qiskit
print(qiskit.version.get_version_info())

2.3.0


# VRP initialization

In [2]:
import numpy as np

n = 4  # number of nodes + depot (n+1)
K = 2  # number of vehicles

# Get the data randomly placed nodes
class Initializer:
    def __init__(self, n):
        self.n = n

    def generate_instance(self):
        n = self.n

        np.random.seed(1543)

        xc = (np.random.rand(n) - 0.5) * 10
        yc = (np.random.rand(n) - 0.5) * 10

        instance = np.zeros([n, n])
        for ii in range(0, n):
            for jj in range(ii + 1, n):
                instance[ii, jj] = (xc[ii] - xc[jj]) ** 2 + (yc[ii] - yc[jj]) ** 2
                instance[jj, ii] = instance[ii, jj]

        return xc, yc, instance
    
initializer = Initializer(n)
xc, yc, instance = initializer.generate_instance()

In [3]:
# from qiskit_optimization.problems.quadratic_program import QuadraticProgram
import networkx as nx
import itertools
from docplex.mp.model import Model
from qiskit_optimization.translators import from_docplex_mp
from qiskit_optimization.problems import QuadraticProgram

def vehicle_routing_to_quadratic_program(graph: nx.Graph, depot: int, num_vehicles: int) -> QuadraticProgram:
    """
    Convert a vehicle routing problem instance into a QuadraticProgram.

    Args:
        graph: A networkx graph with edge weights representing distances or costs.
        depot: The index of the depot node.
        num_vehicles: The number of vehicles available.

    Returns:
        A QuadraticProgram representing the vehicle routing problem.
    """
    mdl = Model(name="Vehicle routing")
    n = graph.number_of_nodes()
    x = {}

    # Binary variables: x_ij = 1 if vehicle goes from node i to node j
    for i in range(n):
        for j in range(n):
            if i != j:
                x[(i, j)] = mdl.binary_var(name=f"x_{i}_{j}")

    # Objective: Minimize total travel cost
    mdl.minimize(
        mdl.sum(
            graph.edges[i, j]["weight"] * x[(i, j)]
            for i in range(n)
            for j in range(n)
            if i != j
        )
    )

    # Each node (except depot) has one outgoing and one incoming edge
    for i in range(n):
        if i != depot:
            mdl.add_constraint(mdl.sum(x[i, j] for j in range(n) if i != j) == 1)
    for j in range(n):
        if j != depot:
            mdl.add_constraint(mdl.sum(x[i, j] for i in range(n) if i != j) == 1)

    # Depot constraints: exactly num_vehicles enter and leave
    mdl.add_constraint(
        mdl.sum(x[i, depot] for i in range(n) if i != depot) == num_vehicles
    )
    mdl.add_constraint(
        mdl.sum(x[depot, j] for j in range(n) if j != depot) == num_vehicles
    )

    # Sub-tour elimination (using clique constraints)
    node_list = [i for i in range(n) if i != depot]
    for i in range(2, len(node_list) + 1):
        for clique in itertools.combinations(node_list, i):
            mdl.add_constraint(
                mdl.sum(x[i, j] for i in clique for j in clique if i != j) <= len(clique) - 1
            )

    # Convert to QuadraticProgram
    qp = from_docplex_mp(mdl)
    return qp


def vehicle_routing_to_objective(graph: nx.Graph, depot: int, num_vehicles: int) -> QuadraticProgram:
    mdl = Model(name="Vehicle routing obj")
    n = graph.number_of_nodes()
    x = {}
    # Binary variables: x_ij = 1 if vehicle goes from node i to node j
    for i in range(n):
        for j in range(n):
            if i != j:
                x[(i, j)] = mdl.binary_var(name=f"x_{i}_{j}")
    # Objective: Minimize total travel cost
    mdl.minimize(
        mdl.sum(
            graph.edges[i, j]["weight"] * x[(i, j)]
            for i in range(n)
            for j in range(n)
            if i != j
        )
    )
    # Convert to QuadraticProgram
    qp = from_docplex_mp(mdl)
    return qp
    

In [4]:
import matplotlib.pyplot as plt

G = nx.Graph()

# Add nodes and edges from the distance matrix
num_nodes = instance.shape[0]
for i in range(num_nodes):
    for j in range(i + 1, num_nodes):  # Avoid duplicate edges
        if instance[i, j] > 0:  # skip zero or invalid distances
            G.add_edge(i, j, weight=instance[i, j])

vrp3 = vehicle_routing_to_quadratic_program(G, depot=0, num_vehicles=K)

vrp_obj = vehicle_routing_to_objective(G, depot=0, num_vehicles=K)
# vrp_con = vehicle_routing_to_constraints(G, depot=0, num_vehicles=K)

# q3 = vrp3.to_quadratic_program()
print(vrp3.prettyprint())
vrp_variables = n**2-n

Problem name: Vehicle routing

Minimize
  36.840230524848785*x_0_1 + 5.061352996006702*x_0_2 + 30.631504139776087*x_0_3
  + 36.840230524848785*x_1_0 + 24.553229038971544*x_1_2
  + 63.21893592648173*x_1_3 + 5.061352996006702*x_2_0 + 24.553229038971544*x_2_1
  + 15.497198766817078*x_2_3 + 30.631504139776087*x_3_0
  + 63.21893592648173*x_3_1 + 15.497198766817078*x_3_2

Subject to
  Linear constraints (12)
    x_1_0 + x_1_2 + x_1_3 == 1  'c0'
    x_2_0 + x_2_1 + x_2_3 == 1  'c1'
    x_3_0 + x_3_1 + x_3_2 == 1  'c2'
    x_0_1 + x_2_1 + x_3_1 == 1  'c3'
    x_0_2 + x_1_2 + x_3_2 == 1  'c4'
    x_0_3 + x_1_3 + x_2_3 == 1  'c5'
    x_1_0 + x_2_0 + x_3_0 == 2  'c6'
    x_0_1 + x_0_2 + x_0_3 == 2  'c7'
    x_1_2 + x_2_1 <= 1  'c8'
    x_1_3 + x_3_1 <= 1  'c9'
    x_2_3 + x_3_2 <= 1  'c10'
    x_1_2 + x_1_3 + x_2_1 + x_2_3 + x_3_1 + x_3_2 <= 2  'c11'

  Binary variables (12)
    x_0_1 x_0_2 x_0_3 x_1_0 x_1_2 x_1_3 x_2_0 x_2_1 x_2_3 x_3_0 x_3_1 x_3_2



In [5]:
from qiskit_optimization.converters import QuadraticProgramToQubo

# conv = QuadraticProgramToQubo()

conv = QuadraticProgramToQubo(penalty=np.sum(instance)*2)    # ..............PENALTY................ penalty=np.sum(instance)
qubo_with_constraints = conv.convert(vrp3)
print(qubo_with_constraints.export_as_lp_string())
#print(conv.penalty)
print("\n ----------------------------------- \n")
# qubo_without_constraints = conv0.convert(vrp3)
# print(qubo_without_constraints.export_as_lp_string())
qubo_obj = conv.convert(vrp_obj)

/tmp/ipykernel_2592629/2199314941.py:7: DeprecationWarning: The method ``qiskit_optimization.problems.quadratic_program.QuadraticProgram.export_as_lp_string()`` is deprecated as of Qiskit 0.7.0. It will be removed no earlier than 3 months after the release date. Use prettyprint instead.
  print(qubo_with_constraints.export_as_lp_string())


\ This file has been generated by DOcplex
\ ENCODING=ISO-8859-1
\Problem name: Vehicle routing

Minimize
 obj: - 4182.418602904797 x_0_1 - 4214.197480433639 x_0_2
      - 4188.627329289870 x_0_3 - 4182.418602904797 x_1_0
      - 5601.125215533890 x_1_2 - 5562.459508646380 x_1_3
      - 4214.197480433639 x_2_0 - 5601.125215533890 x_2_1
      - 5610.181245806044 x_2_3 - 4188.627329289870 x_3_0
      - 5562.459508646380 x_3_1 - 5610.181245806044 x_3_2
      - 2812.839222286430 c11@int_slack@0 - 2812.839222286430 c11@int_slack@1 +
      [ 2812.839222286430 x_0_1^2 + 2812.839222286430 x_0_1*x_0_2
      + 2812.839222286430 x_0_1*x_0_3 + 2812.839222286430 x_0_1*x_2_1
      + 2812.839222286430 x_0_1*x_3_1 + 2812.839222286430 x_0_2^2
      + 2812.839222286430 x_0_2*x_0_3 + 2812.839222286430 x_0_2*x_1_2
      + 2812.839222286430 x_0_2*x_3_2 + 2812.839222286430 x_0_3^2
      + 2812.839222286430 x_0_3*x_1_3 + 2812.839222286430 x_0_3*x_2_3
      + 2812.839222286430 x_1_0^2 + 2812.839222286430 x_1_0

In [6]:
Q = qubo_with_constraints.objective.quadratic.to_array()

# Simplify diagonal terms: x^2 = x
# So move diagonal elements to linear and zero them out
linear = qubo_with_constraints.objective.linear.to_array()
diag = np.diag(Q)

# Update linear: x^2 term becomes x
simplified_linear = linear + diag

# Zero out the diagonal in the quadratic matrix
np.fill_diagonal(Q, 0)

# Optional: print simplified objective
# print("Simplified linear:", simplified_linear)
# print("Simplified quadratic (off-diagonal):\n", Q)

In [7]:
def build_qubo_from_arrays(linear, quadratic, variable_names=None):
    n = len(linear)
    qp = QuadraticProgram()
    
    # If variable names are not given, create default names
    if variable_names is None:
        variable_names = [f'x{i}' for i in range(n)]
    
    # Add binary variables
    for name in variable_names:
        qp.binary_var(name._name)
    
    # Add objective: minimize linear + quadratic terms
    linear_dict = {variable_names[i]._name: linear[i] for i in range(n)}
    
    # Only use upper triangle of Q to avoid double-counting
    quadratic_dict = {}
    for i in range(n):
        for j in range(i + 1, n):
            if quadratic[i, j] != 0:
                quadratic_dict[(variable_names[i]._name, variable_names[j]._name)] = quadratic[i, j]

    max_lin_coeff = max(abs(v) for v in linear_dict.values())
    max_quad_coeff = max(abs(v) for v in quadratic_dict.values())

    if max_lin_coeff > max_quad_coeff:
        linear_dict_normed = {k: v / max_lin_coeff for k, v in linear_dict.items()}
        quadratic_dict_normed = {k: v / max_lin_coeff for k, v in quadratic_dict.items()}
    else:
        linear_dict_normed = {k: v / max_quad_coeff for k, v in linear_dict.items()}
        quadratic_dict_normed = {k: v / max_quad_coeff for k, v in quadratic_dict.items()}         
    
    qp.minimize(linear=linear_dict_normed, quadratic=quadratic_dict_normed)
    
    return qp

In [8]:
qubo_simple = build_qubo_from_arrays(simplified_linear, Q, variable_names=qubo_with_constraints.variables)
print(qubo_simple.export_as_lp_string())

/tmp/ipykernel_2592629/3128702919.py:2: DeprecationWarning: The method ``qiskit_optimization.problems.quadratic_program.QuadraticProgram.export_as_lp_string()`` is deprecated as of Qiskit 0.7.0. It will be removed no earlier than 3 months after the release date. Use prettyprint instead.
  print(qubo_simple.export_as_lp_string())


\ This file has been generated by DOcplex
\ ENCODING=ISO-8859-1
\Problem name: CPLEX

Minimize
 obj: - 0.793017537604 x_0_1 - 0.802095785572 x_0_2 - 0.794791179786 x_0_3
      - 0.793017537604 x_1_0 - 0.997412970665 x_1_2 - 0.986367367350 x_1_3
      - 0.802095785572 x_2_0 - 0.997412970665 x_2_1 - x_2_3
      - 0.794791179786 x_3_0 - 0.986367367350 x_3_1 - x_3_2
      - 0.602656243848 c11@int_slack@0 - 0.602656243848 c11@int_slack@1 + [
      0.803541658464 x_0_1*x_0_2 + 0.803541658464 x_0_1*x_0_3
      + 0.803541658464 x_0_1*x_2_1 + 0.803541658464 x_0_1*x_3_1
      + 0.803541658464 x_0_2*x_0_3 + 0.803541658464 x_0_2*x_1_2
      + 0.803541658464 x_0_2*x_3_2 + 0.803541658464 x_0_3*x_1_3
      + 0.803541658464 x_0_3*x_2_3 + 0.803541658464 x_1_0*x_1_2
      + 0.803541658464 x_1_0*x_1_3 + 0.803541658464 x_1_0*x_2_0
      + 0.803541658464 x_1_0*x_3_0 + 1.607083316928 x_1_2*x_1_3
      + 1.205312487696 x_1_2*x_2_1 + 0.803541658464 x_1_2*x_2_3
      + 0.803541658464 x_1_2*x_3_1 + 1.6070833169

In [9]:
num_variables = 0
for nam in qubo_with_constraints.variables:
    num_variables +=1
    #print(nam._name)

num_qubits = num_variables
print(num_qubits)

14


In [11]:
def calculate_cost(state, qubo):
    n = len(qubo.variables)
    x = np.fromiter(((state >> i) & 1 for i in range(n)), dtype=np.int8, count=n)
    return qubo.objective.evaluate(x)

In [12]:
ising_H = qubo_simple.to_ising()[0]

In [13]:
from qiskit.quantum_info import SparsePauliOp

def filter_adjacent_ZZ(op: SparsePauliOp) -> SparsePauliOp:
    """
    Keep only terms with ZZ on consecutive qubits in a SparsePauliOp.
    All other terms are discarded.
    """
    new_paulis = []
    new_coeffs = []

    for pauli, coeff in zip(op.paulis, op.coeffs):
        # Identify positions where this term has a Z
        z_positions = [i for i, p in enumerate(pauli.to_label()) if p == 'Z']
        
        # Case: exactly two Zs, and they are adjacent
        if len(z_positions) == 2 and abs(z_positions[0] - z_positions[1]) == 1:
            new_paulis.append(pauli)
            new_coeffs.append(coeff)
        # Keep identity or single-Z terms if you want
        elif len(z_positions) <= 1:
            new_paulis.append(pauli)
            new_coeffs.append(coeff)
        # Drop everything else

    return SparsePauliOp(new_paulis, coeffs=new_coeffs)

In [14]:
ising_H_LC = filter_adjacent_ZZ(ising_H)
print(ising_H_LC)

SparsePauliOp(['IIIIIIIIIIIIIZ', 'IIIIIIIIIIIIZI', 'IIIIIIIIIIIZII', 'IIIIIIIIIIZIII', 'IIIIIIIIIZIIII', 'IIIIIIIIZIIIII', 'IIIIIIIZIIIIII', 'IIIIIIZIIIIIII', 'IIIIIZIIIIIIII', 'IIIIZIIIIIIIII', 'IIIZIIIIIIIIII', 'IIZIIIIIIIIIII', 'IZIIIIIIIIIIII', 'ZIIIIIIIIIIIII', 'IIIIIIIIIIIIZZ', 'IIIIIIIIIIIZZI', 'IIIIIIIIIZZIII', 'IIIIIIIIZZIIII', 'IIIIIIZZIIIIII', 'IIIIIZZIIIIIII', 'IIIZZIIIIIIIII', 'IIZZIIIIIIIIII', 'IZZIIIIIIIIIII', 'ZZIIIIIIIIIIII'],
              coeffs=[-0.00526206+0.j, -0.00072294+0.j, -0.00437524+0.j, -0.00526206+0.j,
 -0.65638465+0.j, -0.66190745+0.j, -0.00072294+0.j, -0.65638465+0.j,
 -0.65509113+0.j, -0.00437524+0.j, -0.66190745+0.j, -0.65509113+0.j,
 -0.40177083+0.j, -0.40177083+0.j,  0.10044271+0.j,  0.10044271+0.j,
  0.10044271+0.j,  0.20088541+0.j,  0.10044271+0.j,  0.20088541+0.j,
  0.10044271+0.j,  0.20088541+0.j,  0.10044271+0.j,  0.10044271+0.j])


# AQC stuff

In [15]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Callable, Dict, Iterable, Optional, Tuple

import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp
from qiskit.circuit.library import PauliEvolutionGate


# ----------------------------
# Annealing circuit utilities
# ----------------------------

def default_schedule_s(step_index: int, num_steps: int) -> float:
    """
    Linear schedule s in [0, 1] at the *start* of step `step_index`.
    If you prefer midpoint-rule, use `use_midpoint=True` in builders below.
    """
    if num_steps <= 0:
        raise ValueError("num_steps must be positive.")
    return step_index / num_steps


def build_driver_x_hamiltonian(num_qubits: int, coeff: float = 1.0) -> SparsePauliOp:
    """H_initial = coeff * sum_i X_i."""
    terms = []
    for i in range(num_qubits):
        pauli = ["I"] * num_qubits
        pauli[i] = "X"
        terms.append("".join(pauli[::-1]))  # Qiskit Pauli strings are right-to-left
    return SparsePauliOp(terms, coeffs=[coeff] * len(terms))


def build_annealing_segment(
    num_qubits: int,
    H_initial: SparsePauliOp,
    H_problem: SparsePauliOp,
    delta_t: float,
    step_start: int,
    step_end: int,
    *,
    schedule_s: Callable[[int, int], float],
    num_steps_total: int,
    s_scale: float = 1.0,
    use_midpoint: bool = False,
    insert_barriers: bool = False,
) -> QuantumCircuit:
    """
    Builds steps [step_start, step_end) of annealing evolution:
        U_k ≈ exp(-i * ((1-s)*H_initial + s*H_problem) * delta_t)
    where s = s_scale * schedule_s(k or k+0.5, num_steps_total).

    Note: if you literally want your current behavior (s *= pi), set s_scale=np.pi.
    """
    if not (0 <= step_start <= step_end <= num_steps_total):
        raise ValueError("Require 0 <= step_start <= step_end <= num_steps_total.")
    if delta_t <= 0:
        raise ValueError("delta_t must be positive.")

    qc = QuantumCircuit(num_qubits)

    for k in range(step_start, step_end):
        k_eff = (k + 0.5) if use_midpoint else k
        s = float(schedule_s(k_eff, num_steps_total)) * float(s_scale)

        # If you want to clamp s into [0,1], uncomment:
        # s = max(0.0, min(1.0, s))

        H_t = (1.0 - s) * H_initial + s * H_problem
        qc.append(PauliEvolutionGate(H_t, time=delta_t), range(num_qubits))

        if insert_barriers:
            qc.barrier()

    return qc


def build_full_trotter_annealing_circuit(
    num_qubits: int,
    H_problem: SparsePauliOp,
    *,
    num_steps: int,
    delta_t: Optional[float] = None,
    H_initial: Optional[SparsePauliOp] = None,
    schedule_s: Callable[[int, int], float] = default_schedule_s,
    s_scale: float = 1.0,
    use_midpoint: bool = False,
    measure: bool = True,
) -> QuantumCircuit:
    """
    Full trotterized annealing circuit with |+...+> initialization.
    """
    if num_steps <= 0:
        raise ValueError("num_steps must be positive.")
    if delta_t is None:
        delta_t = 1.0 / num_steps

    if H_initial is None:
        H_initial = build_driver_x_hamiltonian(num_qubits)

    qc = QuantumCircuit(num_qubits)
    qc.h(range(num_qubits))
    qc.barrier()

    seg = build_annealing_segment(
        num_qubits,
        H_initial,
        H_problem,
        delta_t,
        0,
        num_steps,
        schedule_s=schedule_s,
        num_steps_total=num_steps,
        s_scale=s_scale,
        use_midpoint=use_midpoint,
        insert_barriers=False,
    )
    qc.compose(seg, inplace=True)

    qc.barrier()
    if measure:
        qc.measure_all()
    return qc


# ----------------------------
# AQC-Tensor compression
# ----------------------------

@dataclass
class AQCSettings:
    """
    Settings for AQC-Tensor optimization (prefix compression).

    Requires:
      pip install 'qiskit-addon-aqc-tensor[aer,quimb-jax]'
    as in IBM tutorial. :contentReference[oaicite:1]{index=1}
    """
    stopping_fidelity: float = 0.999  # stop when fidelity >= this
    maxiter: int = 300
    method: str = "L-BFGS-B"


def aqc_tensor_compress_prefix(
    target_prefix_circuit: QuantumCircuit,
    good_prefix_circuit: QuantumCircuit,
    *,
    aqc_settings: AQCSettings = AQCSettings(),
) -> QuantumCircuit:
    """
    Compress `target_prefix_circuit` into a low-depth ansatz initialized from `good_prefix_circuit`,
    optimizing state fidelity using the AQC-Tensor workflow. :contentReference[oaicite:2]{index=2}
    """
    
    import datetime
    import quimb.tensor
    from scipy.optimize import OptimizeResult, minimize

    from qiskit_addon_aqc_tensor.ansatz_generation import generate_ansatz_from_circuit
    from qiskit_addon_aqc_tensor.objective import MaximizeStateFidelity
    from qiskit_addon_aqc_tensor.simulation.quimb import QuimbSimulator
    from qiskit_addon_aqc_tensor.simulation import tensornetwork_from_circuit
    from qiskit_aer import AerSimulator

    # Simulator settings 
    # simulator_settings = QuimbSimulator(quimb.tensor.CircuitMPS, autodiff_backend="jax")
    simulator_settings = AerSimulator(method="matrix_product_state",matrix_product_state_max_bond_dimension=100)

    # Build target MPS from the (deep) target prefix
    target_mps = tensornetwork_from_circuit(target_prefix_circuit, simulator_settings)

    # Generate parametrized ansatz + initialization from a shallower "good" prefix
    ansatz, x0 = generate_ansatz_from_circuit(good_prefix_circuit.decompose(reps=4))

    # Optimize parameters to maximize fidelity with target MPS
    stopping_point = 1.0 - float(aqc_settings.stopping_fidelity)
    objective = MaximizeStateFidelity(target_mps, ansatz, simulator_settings)

    def callback(intermediate_result: OptimizeResult):
        fidelity = 1.0 - intermediate_result.fun
        # You can comment this out if you want it quiet:
        print(f"{datetime.datetime.now()}  fidelity={fidelity:.8f}")
        if intermediate_result.fun < stopping_point:
            raise StopIteration

    res = minimize(
        objective,
        x0,
        method=aqc_settings.method,
        jac=True,
        options={"maxiter": int(aqc_settings.maxiter)},
        callback=callback,
    )

    # Accept: success, maxiter, or early StopIteration termination
    if res.status not in (0, 1, 99):
        raise RuntimeError(f"AQC optimization failed: {res.message} (status={res.status})")

    x_final = res.x
    compressed_prefix = ansatz.assign_parameters(x_final, inplace=False)
    return compressed_prefix

def build_annealing_circuit_with_aqc_prefix(
    num_qubits: int,
    H_problem: SparsePauliOp,
    *,
    num_steps: int,
    alpha: int,
    delta_t: Optional[float] = None,
    H_initial: Optional[SparsePauliOp] = None,
    schedule_s: Callable[[int, int], float] = default_schedule_s,
    s_scale: float = 1.0,
    use_midpoint: bool = False,
    aqc_target_refinement: int = 1,
    aqc_good_steps: Optional[int] = None,
    aqc_settings: AQCSettings = AQCSettings(),
    measure: bool = True,
):
    """
    Returns:
        qc: full circuit
        prefix_dict: dict of prefix-related circuits
        remaining_dict: dict of post-prefix / remaining circuits
    """
    if num_steps <= 0:
        raise ValueError("num_steps must be positive.")
    if not (0 <= alpha <= num_steps):
        raise ValueError("alpha must satisfy 0 <= alpha <= num_steps.")
    if delta_t is None:
        delta_t = 1.0 / num_steps
    if H_initial is None:
        H_initial = build_driver_x_hamiltonian(num_qubits)

    prefix_dict = {
        "target_prefix": None,
        "good_prefix": None,
        "compressed_prefix": None,
    }

    remaining_dict = {
        "tail": None,
        "initial_hadamards": None,
        "full_prefix_plus_tail_no_measure": None,
    }

    qc = QuantumCircuit(num_qubits)

    # Initial state prep
    init_circ = QuantumCircuit(num_qubits)
    init_circ.h(range(num_qubits))
    remaining_dict["initial_hadamards"] = init_circ.copy()

    qc.compose(init_circ, inplace=True)
    qc.barrier()

    # --- Build/compress prefix ---
    if alpha > 0:
        if aqc_target_refinement < 1:
            raise ValueError("aqc_target_refinement must be >= 1.")

        refined_steps = alpha * aqc_target_refinement
        refined_dt = delta_t / aqc_target_refinement

        target_prefix = QuantumCircuit(num_qubits)
        for k in range(refined_steps):
            t_frac = (k + (0.5 if use_midpoint else 0.0)) / refined_steps
            k_eff = t_frac * alpha
            s = float(schedule_s(k_eff, num_steps)) * float(s_scale)
            H_t = (1.0 - s) * H_initial + s * H_problem
            target_prefix.append(PauliEvolutionGate(H_t, time=refined_dt), range(num_qubits))

        total_prefix_time = alpha * delta_t
        if aqc_good_steps is None:
            aqc_good_steps = max(1, int(np.ceil(alpha / 3)))

        good_prefix = QuantumCircuit(num_qubits)
        good_dt = total_prefix_time / aqc_good_steps
        for j in range(aqc_good_steps):
            t_frac = (j + (0.5 if use_midpoint else 0.0)) / aqc_good_steps
            k_eff = t_frac * alpha
            s = float(schedule_s(k_eff, num_steps)) * float(s_scale)
            H_t = (1.0 - s) * H_initial + s * H_problem
            good_prefix.append(PauliEvolutionGate(H_t, time=good_dt), range(num_qubits))

        compressed_prefix = aqc_tensor_compress_prefix(
            target_prefix.decompose(reps=4), good_prefix, aqc_settings=aqc_settings
        )

        prefix_dict["target_prefix"] = target_prefix
        prefix_dict["good_prefix"] = good_prefix
        prefix_dict["compressed_prefix"] = compressed_prefix

        qc.compose(compressed_prefix, inplace=True)
        qc.barrier()

    # --- Tail / remaining evolution ---
    if alpha < num_steps:
        tail = build_annealing_segment(
            num_qubits,
            H_initial,
            H_problem,
            delta_t,
            alpha,
            num_steps,
            schedule_s=schedule_s,
            num_steps_total=num_steps,
            s_scale=s_scale,
            use_midpoint=use_midpoint,
        )
        remaining_dict["tail"] = tail
        qc.compose(tail, inplace=True)

    qc.barrier()

    if measure:
        qc.measure_all()

    # store a no-measure full evolution circuit
    full_no_measure = QuantumCircuit(num_qubits)
    full_no_measure.compose(init_circ, inplace=True)
    if prefix_dict["compressed_prefix"] is not None:
        full_no_measure.compose(prefix_dict["compressed_prefix"], inplace=True)
    if remaining_dict["tail"] is not None:
        full_no_measure.compose(remaining_dict["tail"], inplace=True)
    remaining_dict["full_prefix_plus_tail_no_measure"] = full_no_measure

    return qc, prefix_dict, remaining_dict


def make_circuits_for_alphas(
    alphas: Iterable[int],
    num_qubits: int,
    H_problem: SparsePauliOp,
    *,
    num_steps: int,
    delta_t: Optional[float] = None,
    H_initial: Optional[SparsePauliOp] = None,
    schedule_s: Callable[[int, int], float] = default_schedule_s,
    s_scale: float = 1.0,
    use_midpoint: bool = False,
    aqc_target_refinement: int = 1,
    aqc_good_steps: Optional[int] = None,
    aqc_settings: AQCSettings = AQCSettings(),
    measure: bool = True,
) -> Tuple[Dict[int, QuantumCircuit], Dict[int, Dict[str, Optional[QuantumCircuit]]], Dict[int, Dict[str, Optional[QuantumCircuit]]]]:
    """
    Returns:
        circuits_by_alpha: {alpha: full circuit}
        prefixes_by_alpha: {alpha: prefix_dict}
        remaining_by_alpha: {alpha: remaining_dict}
    """
    circuits_by_alpha: Dict[int, QuantumCircuit] = {}
    prefixes_by_alpha: Dict[int, Dict[str, Optional[QuantumCircuit]]] = {}
    remaining_by_alpha: Dict[int, Dict[str, Optional[QuantumCircuit]]] = {}

    for a in alphas:
        a_int = int(a)
        print(a_int)

        qc, prefix_dict, remaining_dict = build_annealing_circuit_with_aqc_prefix(
            num_qubits,
            H_problem,
            num_steps=num_steps,
            alpha=a_int,
            delta_t=delta_t,
            H_initial=H_initial,
            schedule_s=schedule_s,
            s_scale=s_scale,
            use_midpoint=use_midpoint,
            aqc_target_refinement=aqc_target_refinement,
            aqc_good_steps=aqc_good_steps,
            aqc_settings=aqc_settings,
            measure=measure,
        )

        circuits_by_alpha[a_int] = qc
        prefixes_by_alpha[a_int] = prefix_dict
        remaining_by_alpha[a_int] = remaining_dict

    return circuits_by_alpha, prefixes_by_alpha, remaining_by_alpha



In [71]:
import numpy as np
import pandas as pd
from typing import Dict, Optional, Any, List
from qiskit import QuantumCircuit


def get_meas_counts_from_sampler_result(pub_result) -> Dict[str, int]:
    """
    Extract bitstring->counts from SamplerV2-style result[0].data.meas.
    Adjust this only if your sampler output format differs.
    """
    meas = pub_result.data.meas

    # Most common in your earlier code
    if hasattr(meas, "get_counts"):
        return meas.get_counts()

    # Fallbacks if needed
    if hasattr(meas, "get_int_counts"):
        int_counts = meas.get_int_counts()
        if len(int_counts) == 0:
            return {}
        nbits = max(int(k).bit_length() for k in int_counts.keys())
        return {format(int(k), f"0{nbits}b"): v for k, v in int_counts.items()}

    raise ValueError("Could not extract counts from sampler result.")



def analyze_counts_for_vrp(
    counts: Dict[str, int],
    vrp3,
    num_variables: int,
    shots: Optional[int] = None,
) -> Dict[str, Any]:
    """
    Given measurement counts, compute:
      - feasible bitstring percentage (unique feasible bitstrings / unique observed bitstrings)
      - feasible counts percentage (shots landing on feasible solutions / total shots)
      - best feasible bitstring and its rank by counts
      - optionally also best overall sampled bitstring (feasible or infeasible)
    """
    if not counts:
        return {
            "num_unique_bitstrings": 0,
            "num_feasible_bitstrings": 0,
            "feasible_bitstrings_pct": np.nan,
            "feasible_counts": 0,
            "feasible_counts_pct": np.nan,
            "best_feasible_bitstring": None,
            "best_feasible_cost": np.nan,
            "best_feasible_bitstring_count": 0,
            "best_feasible_bitstring_shot_pct": np.nan,
            "best_feasible_bitstring_rank_by_count": np.nan,
            "best_overall_bitstring": None,
            "best_overall_cost": np.nan,
            "best_overall_feasible": np.nan,
        }

    if shots is None:
        shots = int(sum(counts.values()))

    total_unique = len(counts)
    feasible_unique = 0
    feasible_counts = 0

    bitstring_records = []

    sorted_items = sorted(counts.items(), key=lambda kv: kv[1], reverse=True)

    for rank_idx, (bs, ct) in enumerate(sorted_items, start=1):
        x = np.array([int(b) for b in bs[-num_variables:][::-1]])
        cost = vrp3.objective.evaluate(x)
        feasible = vrp3.is_feasible(x)

        if feasible:
            feasible_unique += 1
            feasible_counts += ct

        bitstring_records.append(
            {
                "bitstring": bs,
                "count": int(ct),
                "rank_by_count": rank_idx,
                "cost": float(cost),
                "feasible": bool(feasible),
            }
        )

    # best overall sampled string (feasible OR infeasible)
    best_overall = min(bitstring_records, key=lambda r: r["cost"])

    # best feasible sampled string only
    feasible_records = [r for r in bitstring_records if r["feasible"]]
    if feasible_records:
        best_feasible = min(feasible_records, key=lambda r: r["cost"])
    else:
        best_feasible = None

    return {
        "num_unique_bitstrings": total_unique,
        "num_feasible_bitstrings": feasible_unique,
        "feasible_bitstrings_pct": feasible_unique / total_unique if total_unique > 0 else np.nan,
        "feasible_counts": feasible_counts,
        "feasible_counts_pct": feasible_counts / shots if shots > 0 else np.nan,

        "best_feasible_bitstring": None if best_feasible is None else best_feasible["bitstring"],
        "best_feasible_cost": np.nan if best_feasible is None else best_feasible["cost"],
        "best_feasible_bitstring_count": 0 if best_feasible is None else best_feasible["count"],
        "best_feasible_bitstring_shot_pct": (
            np.nan if best_feasible is None or shots <= 0 else best_feasible["count"] / shots
        ),
        "best_feasible_bitstring_rank_by_count": (
            np.nan if best_feasible is None else best_feasible["rank_by_count"]
        ),

        "best_overall_bitstring": best_overall["bitstring"],
        "best_overall_cost": best_overall["cost"],
        "best_overall_feasible": best_overall["feasible"],
    }


def benchmark_alpha_circuits_to_csv(
    circuits_by_alpha: Dict[int, QuantumCircuit],
    prefixes_by_alpha: Dict[int, Dict[str, Optional[QuantumCircuit]]],
    sampler,
    pm,
    vrp3,
    num_variables: int,
    csv_path: str,
    shots: int = 10000,
    prefix_key: str = "compressed_prefix",
    extra_columns: Optional[Dict[str, Any]] = None,
) -> pd.DataFrame:
    """
    For each alpha:
      - transpile prefix and full circuit
      - measure 2-qubit depth of each
      - run full circuit
      - compute feasibility and best-cost stats
      - save cumulative CSV after each alpha

    Parameters
    ----------
    circuits_by_alpha
        dict like {alpha: full_circuit}
    prefixes_by_alpha
        dict like {alpha: {"compressed_prefix": qc, ...}}
    sampler
        Qiskit Runtime SamplerV2 (or compatible)
    pm
        pass manager already created by user
    vrp3
        problem object with:
            vrp3.objective.evaluate(x)
            vrp3.is_feasible(x)
    num_variables
        number of optimization binary variables
    csv_path
        path to csv file to write incrementally
    shots
        sampler shots
    prefix_key
        which prefix to evaluate depth on
    extra_columns
        optional constant metadata appended to every row
    """
    rows: List[Dict[str, Any]] = []
    alpha_list = sorted(circuits_by_alpha.keys())

    for alpha in alpha_list:
        print(f"Running alpha={alpha}")

        full_qc = circuits_by_alpha[alpha]
        prefix_qc = prefixes_by_alpha[alpha].get(prefix_key, None)

        # transpile full circuit
        full_qc_t = pm.run(full_qc)
        full_two_qubit_depth = full_qc_t.depth(lambda x: x.operation.num_qubits == 2)

        # transpile prefix circuit if present
        if prefix_qc is not None and len(prefix_qc.data) > 0:
            prefix_qc_t = pm.run(prefix_qc)
            prefix_two_qubit_depth = prefix_qc_t.depth(lambda x: x.operation.num_qubits == 2)
        else:
            prefix_two_qubit_depth = 0

        # run full circuit
        job = sampler.run([(full_qc_t,)], shots=int(shots))
        pub_result = job.result()[0]
        counts = get_meas_counts_from_sampler_result(pub_result)

        # analyze outputs
        metrics = analyze_counts_for_vrp(
            counts=counts,
            vrp3=vrp3,
            num_variables=num_variables,
            shots=shots,
        )

        row = {
            "alpha": int(alpha),
            "shots": int(shots),
            "prefix_two_qubit_depth": prefix_two_qubit_depth,
            "full_two_qubit_depth": full_two_qubit_depth,
            **metrics,
        }

        if extra_columns is not None:
            row.update(extra_columns)

        rows.append(row)

        # save incrementally after each run
        df_partial = pd.DataFrame(rows)
        df_partial.to_csv(csv_path, index=False)

    df = pd.DataFrame(rows)
    return df

In [72]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit import generate_preset_pass_manager
service = QiskitRuntimeService(name="SAVED_SERVICE_NAME")

backend = service.backend("ibm_rensselaer")

pm = generate_preset_pass_manager(optimization_level=3, backend=backend)

from qiskit_ibm_runtime import SamplerV2 as Sampler
# from qiskit_aer.primitives import SamplerV2 as Sampler
# sampler = Sampler()
sampler = Sampler(mode=backend)
sampler.options.default_shots = 10000
# # Set simple error suppression/mitigation options
sampler.options.dynamical_decoupling.enable = True
sampler.options.dynamical_decoupling.sequence_type = "XpXm"

In [18]:
num_steps = 10
delta_t = 1/num_steps
H_problem = ising_H
H_initial = build_driver_x_hamiltonian(num_qubits)
alphas = [0, 1, 2, 3, 4, 5, 6]
p_list = [1,2,3,4,5,6]

In [19]:
circuits, prefixes, remaining = make_circuits_for_alphas(
    alphas=alphas,
    num_qubits=num_qubits,
    H_problem=ising_H,
    num_steps=num_steps,
    H_initial=H_initial,
    use_midpoint=True,
    aqc_target_refinement = 2,
)

0
1
2026-03-12 14:41:17.206511  fidelity=0.99999999
2
2026-03-12 14:41:24.490453  fidelity=0.99999835
3
2026-03-12 14:41:32.321358  fidelity=0.99998034
4
2026-03-12 14:41:53.218244  fidelity=0.99998382
5
2026-03-12 14:42:20.573839  fidelity=0.99994162
6
2026-03-12 14:42:51.676435  fidelity=0.99981200


In [73]:
df_results = benchmark_alpha_circuits_to_csv(
    circuits_by_alpha=circuits,
    prefixes_by_alpha=prefixes,
    sampler=sampler,
    pm=pm,
    vrp3=vrp3,
    num_variables=len(vrp3.variables),
    csv_path="VRP_4n_aqc_alpha_results.csv",
    shots=10000,
    prefix_key="compressed_prefix",
    extra_columns={
        "num_steps": num_steps,
        "n_qubits": circuits[min(circuits.keys())].num_qubits,
    },
)

print(df_results)

Running alpha=0
Running alpha=1
Running alpha=2
Running alpha=3
Running alpha=4
Running alpha=5
Running alpha=6
   alpha  shots  prefix_two_qubit_depth  full_two_qubit_depth  \
0      0  10000                       0                  1014   
1      1  10000                      98                   997   
2      2  10000                     131                  1054   
3      3  10000                     114                   797   
4      4  10000                     204                   799   
5      5  10000                     203                   814   
6      6  10000                     255                   743   

   num_unique_bitstrings  num_feasible_bitstrings  feasible_bitstrings_pct  \
0                   7460                        9                 0.001206   
1                   7541                        8                 0.001061   
2                   7418                        9                 0.001213   
3                   7460                        5      

In [21]:
len(vrp3.variables)


12

In [22]:
from qiskit import QuantumCircuit
from qiskit.circuit import Parameter
from qiskit.circuit.library import RXXGate, RYYGate


def linear_chain_xy_mixer(num_qubits: int, beta=None, param_name: str = "beta") -> QuantumCircuit:
    """
    Build a linear-chain XY mixer circuit

        U_M(beta) = prod_{i=0}^{n-2} exp[-i beta (X_i X_{i+1} + Y_i Y_{i+1})]

    implemented as alternating RXX and RYY rotations on nearest neighbors.

    Args:
        num_qubits: Number of qubits.
        beta: Optional Qiskit Parameter or numeric value. If None, creates Parameter(param_name).
        param_name: Name of parameter if beta is None.

    Returns:
        QuantumCircuit: parameterized linear-chain XY mixer.
    """
    if beta is None:
        beta = Parameter(param_name)

    qc = QuantumCircuit(num_qubits, name="LC_XY_Mixer")

    for i in range(num_qubits - 1):
        qc.append(RXXGate(2 * beta), [i, i + 1])
        qc.append(RYYGate(2 * beta), [i, i + 1])

    return qc

In [23]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import SamplerV2 as Sampler
from qiskit import QuantumCircuit
from qiskit import generate_preset_pass_manager

pm = generate_preset_pass_manager(optimization_level=3, backend=backend)


In [24]:
from qiskit.circuit.library import QAOAAnsatz

def build_prefixed_qaoa(
    prefix_circuit,
    cost_operator,
    mixer,
    p
):
    """
    Creates a QAOAAnsatz using the prefix circuit as initial state.
    """
    qc = QAOAAnsatz(
        cost_operator=cost_operator,
        reps=p,
        initial_state=prefix_circuit,
        mixer_operator=mixer,
        flatten=True
    )
    qc.measure_all()
    return qc

In [54]:
from scipy.optimize import minimize
import numpy as np

def qaoa_sampler_cost_fun(
    params,
    ansatz,
    sampler,
    qubo_simple,
    aggregation=None,
):
    """
    Sampler-based QAOA objective evaluated on a QUBO cost function.

    Args:
        params (np.ndarray): Parameters for the ansatz.
        ansatz (QuantumCircuit): Parameterized QAOA ansatz.
        sampler: Sampler to be used.
        qubo_simple: QUBO data passed to calculate_cost(state, qubo_simple).
        aggregation (Callable | float | None):
            - None  -> mean objective
            - float -> CVaR alpha
            - callable -> custom aggregation
    """
    job = sampler.run([(ansatz, params)])
    sampler_result = job.result()
    
    sampled_int_counts = sampler_result[0].data.meas.get_int_counts()
    shots = sum(sampled_int_counts.values())

    int_count_distribution = {
        key: val / shots for key, val in sampled_int_counts.items()
    }

    evaluated = {
        state: (
            probability,
            calculate_cost(state, qubo_simple),
        )
        for state, probability in int_count_distribution.items()
    }
    
    if aggregation is None:
        result = sum(probability * value for probability, value in evaluated.values())
    elif isinstance(aggregation, float):
        cvar_aggregation = _get_cvar_aggregation(aggregation)
        result = cvar_aggregation(evaluated.values())
    else:
        result = aggregation(evaluated.values())

    return result

def _get_cvar_aggregation(alpha: float | None) -> Callable:
    """Return the CVaR aggregation function with the given alpha.
 
    Args:
        alpha (float | None): Alpha value for the CVaR aggregation. If None, 1 is used
            by default.
    Raises:
        ValueError: If alpha is not in [0, 1].
    """
    if alpha is None:
        alpha = 1
    elif not 0 <= alpha <= 1:
        raise ValueError(f"alpha must be in [0, 1], but {alpha} was given.")
 
    def cvar_aggregation(
        objective_dict: Iterable[tuple[float, float]],
    ) -> float:
        """Return the CVaR of the given measurements.
        Args:
            objective_dict (Iterable[tuple[float, float]]): An iterable of tuples containing
                the measured bit string and the objective value based on the bit string.
 
        """
        sorted_measurements = sorted(objective_dict, key=lambda x: x[1])
        # accumulate the probabilities until alpha is reached
        accumulated_percent = 0.0
        cvar = 0.0
        for probability, value in sorted_measurements:
            cvar += value * min(probability, alpha - accumulated_percent)
            accumulated_percent += probability
            if accumulated_percent >= alpha:
                break
        return cvar / alpha
 
    return cvar_aggregation    

In [26]:
def analyze_counts_int(
    int_counts,
    vrp3,
    num_variables,
    shots,
    qubo_simple,
):
    """
    Analyze final measurement counts.
    int_counts keys are integers.
    """
    feasible_unique = 0
    feasible_counts = 0
    records = []

    sorted_items = sorted(int_counts.items(), key=lambda x: x[1], reverse=True)

    for rank, (state_int, ct) in enumerate(sorted_items, start=1):
        bs = format(state_int, f"0{num_variables}b")
        x = np.array([int(b) for b in bs[-num_variables:][::-1]])

        cost = calculate_cost(state_int, qubo_simple)
        feasible = vrp3.is_feasible(x)

        if feasible:
            feasible_unique += 1
            feasible_counts += ct

        records.append(
            {
                "state_int": int(state_int),
                "bitstring": bs,
                "count": int(ct),
                "cost": float(cost),
                "feasible": bool(feasible),
                "rank": int(rank),
            }
        )

    if len(records) == 0:
        return {
            "num_unique_bitstrings": 0,
            "num_feasible_bitstrings": 0,
            "feasible_bitstrings_pct": np.nan,
            "feasible_counts": 0,
            "feasible_counts_pct": np.nan,
            "best_bitstring": None,
            "best_cost": np.nan,
            "best_bitstring_rank": np.nan,
            "best_bitstring_count": 0,
        }

    best = min(records, key=lambda r: r["cost"])

    return {
        "num_unique_bitstrings": len(records),
        "num_feasible_bitstrings": feasible_unique,
        "feasible_bitstrings_pct": feasible_unique / len(records),
        "feasible_counts": feasible_counts,
        "feasible_counts_pct": feasible_counts / shots,
        "best_bitstring": best["bitstring"],
        "best_cost": best["cost"],
        "best_bitstring_rank": best["rank"],
        "best_bitstring_count": best["count"],
    }

In [53]:
from scipy.optimize import minimize

def optimize_prefixed_qaoa_cvar(
    ansatz,
    sampler,
    qubo_simple,
    aggregation=0.2,
    maxiter=250,
    x0=None,
    method="COBYQA",
):
    """
    Optimize a parameterized prefixed QAOA ansatz using CVaR or mean cost.
    """
    num_params = ansatz.num_parameters

    if x0 is None:
        x0 = np.random.uniform(0.1, 1.1, size=num_params)

    history = {"objective": []}

    def objective(theta):
        val = qaoa_sampler_cost_fun(
            params=np.asarray(theta, dtype=float),
            ansatz=ansatz,
            sampler=sampler,
            qubo_simple=qubo_simple,
            aggregation=aggregation,
        )
        history["objective"].append(val)
        return val

    # print("Objective defined")
    # print("params length ", len(x0))
    res = minimize(
        objective,
        x0=np.asarray(x0, dtype=float),
        method=method,
        options={"maxiter": int(maxiter)},
    )

    return res, history

In [66]:
import pandas as pd

def compute_cvar_aggregation_from_depth(backend, n, two_qubit_gate_depth):
    lf = None
    for el in backend.properties().general:
        if el.name[:2] == "lf" and el.name[3:] == str(n):
            lf = el.value
            break

    if lf is None:
        raise ValueError(f"Could not find layer fidelity lf_{n} in backend.properties().general")

    print("layer fidelity", lf)

    eplg = 1 - lf ** (1 / (n - 1))   # error per layered gate
    fid_cx = 1 - eplg
    gamma_cx = 1 / (fid_cx ** 2)
    gamma_circ = gamma_cx * two_qubit_gate_depth

    cvar_aggregation = 1 / np.sqrt(gamma_circ)          
    return cvar_aggregation, lf, gamma_cx, gamma_circ
    
def run_prefixed_qaoa_cvar_experiment(
    prefixes_by_alpha,
    cost_operator,
    mixer,
    sampler,
    pm,
    vrp3,
    qubo_simple,
    num_variables,
    csv_path,
    qaoa_depths=(1, 2, 3, 4, 5, 6),
    prefix_key="compressed_prefix",
    shots=10000,
    maxiter=250,
    aggregation=0.2,   # CVaR alpha if float
):
    """
    For each alpha and QAOA depth p:
      - build prefixed QAOA ansatz
      - optimize with CVaR objective on QUBO
      - transpile final circuit
      - run final circuit with measurement
      - save metrics to CSV
    """
    rows = []
    p_continue = 0
    
    for alpha in sorted(prefixes_by_alpha.keys()):
        prefix = prefixes_by_alpha[alpha].get(prefix_key, None)
            
        if prefix is None:
            print(f"Skipping alpha={alpha}, missing prefix '{prefix_key}'")
            continue

        for p in qaoa_depths:
            print(f"alpha={alpha}, qaoa_p={p}")

            if mixer == 'xy':
                mixer = linear_chain_xy_mixer(prefix.num_qubits)
            
            # parameterized ansatz without measurements
            ansatz = build_prefixed_qaoa(
                prefix_circuit=prefix,
                cost_operator=cost_operator,
                mixer=mixer,
                p=int(p),
            )
            
            # prefix depth after transpilation
            prefix_t = pm.run(prefix)
            prefix_twoq_depth = prefix_t.depth(lambda x: x.operation.num_qubits == 2)
            
            # transpile parameterized ansatz to estimate actual 2q depth
            # ansatz_for_depth = ansatz.copy()
            # ansatz_for_depth.measure_all()

            #print(ansatz.depth(lambda x: x.operation.num_qubits == 2))
            
            ansatz_t = pm.run(ansatz)
            twoq_depth_for_cvar = ansatz_t.depth(lambda x: x.operation.num_qubits == 2)

            # print("transpiled ", twoq_depth_for_cvar)
            # print(ansatz_t.count_ops())
            
            cvar_aggregation, lf, gamma_cx, gamma_circ = compute_cvar_aggregation_from_depth(
                backend=backend,
                n=ansatz.num_qubits,
                two_qubit_gate_depth=twoq_depth_for_cvar,
            )

            # optimize CVaR / mean QUBO cost
            res, history = optimize_prefixed_qaoa_cvar(
                ansatz=ansatz_t,
                sampler=sampler,
                qubo_simple=qubo_simple,
                aggregation=cvar_aggregation,
                maxiter=maxiter,
            )

            # bind final parameters
            param_list = list(ansatz_t.parameters)
            bind_dict = {param_list[i]: float(res.x[i]) for i in range(len(param_list))}
            qc_final = ansatz_t.assign_parameters(bind_dict, inplace=False)

            # add measurements only for final evaluation
            # qc_eval = qc_final.copy()
            #qc_eval.measure_all()

            # transpile final circuit
            # qc_eval_t = pm.run(qc_eval)
            full_twoq_depth = qc_final.depth(lambda x: x.operation.num_qubits == 2)

            # run final circuit
            job = sampler.run([(qc_final,)], shots=int(shots))
            sampler_result = job.result()
            int_counts = sampler_result[0].data.meas.get_int_counts()

            # final metrics
            metrics = analyze_counts_int(
                int_counts=int_counts,
                vrp3=vrp3,
                num_variables=num_variables,
                shots=shots,
                qubo_simple=qubo_simple,
            )

            row = {
                "alpha": int(alpha),
                "qaoa_p": int(p),
                "aggregation": aggregation,
                "num_iterations": getattr(res, "nfev", np.nan),
                "optimizer_success": getattr(res, "success", False),
                "final_objective": float(res.fun),
                "prefix_twoq_depth": prefix_twoq_depth,
                "full_twoq_depth": full_twoq_depth,
                **metrics,
            }

            rows.append(row)
            pd.DataFrame(rows).to_csv(csv_path, index=False)

    return pd.DataFrame(rows)

In [29]:
# for el in backend.properties().general:
#     if el.name[:2] == "lf" and el.name[3:] == str(n):
#         lf = el.value  # layer fidelity
#         print("layer fidelity", lf)
#         eplg = 1 - lf ** (1 / (n - 1))  # error per layered gate (EPLG)
#         fid_cx = 1 - eplg
#         gamma_cx = 1 / fid_cx**2
#         gamma_circ = gamma_cx * two_qubit_gate_depth
 
# cvar_aggregation = 1 / np.sqrt(gamma_circ)
# print(cvar_aggregation)

In [30]:
df = run_prefixed_qaoa_cvar_experiment(
    prefixes_by_alpha=prefixes,
    cost_operator=ising_H,
    mixer=None,
    sampler=sampler,
    pm=pm,
    vrp3=vrp3,
    qubo_simple=qubo_simple,
    num_variables=len(vrp3.variables),
    csv_path="rensselaer_4n_prefixed_qaoa_cvar_results.csv",
    qaoa_depths=p_list,
    prefix_key="compressed_prefix",
    shots=10000,
    maxiter=250,
    aggregation=1,   
)

In [31]:
df_lc = run_prefixed_qaoa_cvar_experiment(
    prefixes_by_alpha=prefixes,
    cost_operator=ising_H_LC,
    mixer=None,
    sampler=sampler,
    pm=pm,
    vrp3=vrp3,
    qubo_simple=qubo_simple,
    num_variables=len(vrp3.variables),
    csv_path="rensselaer_4nLC_prefixed_qaoa_cvar_results.csv",
    qaoa_depths=p_list,
    prefix_key="compressed_prefix",
    shots=10000,
    maxiter=250,
    aggregation=1,  
)

# Variance 

In [32]:
from qiskit.quantum_info import SparsePauliOp

def frozen_hamiltonian_at_alpha(
    alpha: int,
    num_steps: int,
    H_initial: SparsePauliOp,
    H_problem: SparsePauliOp,
    schedule_s,
    s_scale: float = 1.0,
    use_midpoint: bool = False,
):
    k_eff = (alpha + 0.5) if use_midpoint else alpha
    s = float(schedule_s(k_eff, num_steps)) * float(s_scale)
    return (1.0 - s) * H_initial + s * H_problem

In [33]:
def pauli_square(op: SparsePauliOp) -> SparsePauliOp:
    # Generic operator product using SparsePauliOp composition
    return (op @ op).simplify()

In [34]:
from qiskit_ibm_runtime import EstimatorV2 as Estimator

def measure_prefix_observables_with_estimator(
    estimator,
    prefix,
    pm,
    H_problem: SparsePauliOp,
    H_driver: SparsePauliOp,
    precision: float = 0.02,
):
    """
    Returns <H_problem>, <H_problem^2>, <H_driver>.
    The prefix circuit should be unmeasured.
    """
    H_problem_sq = pauli_square(H_problem)

    prefix_circuit = pm.run(prefix)
    
    observables = [
        H_problem.apply_layout(prefix_circuit.layout),
        H_problem_sq.apply_layout(prefix_circuit.layout),
        H_driver.apply_layout(prefix_circuit.layout),
    ]

    # EstimatorV2 PUB: (circuit, observables)
    # isa_obs = observables.apply_layout(prefix_circuit.layout)
    
    job = estimator.run([(prefix_circuit, observables)], precision=precision)
    result = job.result()[0]

    # V2 returns expectation values in the pub result data container
    evs = np.asarray(result.data.evs, dtype=float).ravel()

    out = {
        "exp_H_problem": float(evs[0]),
        "exp_H_problem_sq": float(evs[1]),
        "var_H_problem": float(evs[1] - evs[0] ** 2),
        "exp_H_driver": float(evs[2]),
    }
    return out

In [35]:
from qiskit import QuantumCircuit
from qiskit.circuit.library import PauliEvolutionGate

def build_probe_circuit(
    prefix_circuit,
    H_frozen: SparsePauliOp,
    V: SparsePauliOp,
    lam: float,
    probe_dt: float,
):
    """
    Returns an unmeasured circuit:
      prefix -> exp(-i probe_dt * (H_frozen + lam * V))
    """
    qc = QuantumCircuit(prefix_circuit.num_qubits)
    qc.compose(prefix_circuit, inplace=True)

    H_probe = (H_frozen + lam * V).simplify()
    qc.append(PauliEvolutionGate(H_probe, time=probe_dt), range(prefix_circuit.num_qubits))
    return qc

In [36]:
def estimate_susceptibility(
    estimator,
    prefix_circuit,
    pm,
    H_frozen: SparsePauliOp,
    V: SparsePauliOp,
    lam: float = 0.02,
    probe_dt: float = 0.05,
    precision: float = 0.02,
):
    qc_plus = build_probe_circuit(prefix_circuit, H_frozen, V, +lam, probe_dt)
    qc_minus = build_probe_circuit(prefix_circuit, H_frozen, V, -lam, probe_dt)

    qc_plus_t = pm.run(qc_plus)
    qc_minus_t = pm.run(qc_minus)

    pubs = [
        (qc_plus_t, [V.apply_layout(qc_plus_t.layout)]),
        (qc_minus_t, [V.apply_layout(qc_minus_t.layout)]),
    ]
    job = estimator.run(pubs, precision=precision)
    res_plus = job.result()[0]
    res_minus = job.result()[1]

    ev_plus = float(np.asarray(res_plus.data.evs).ravel()[0])
    ev_minus = float(np.asarray(res_minus.data.evs).ravel()[0])

    chi = (ev_plus - ev_minus) / (2 * lam)
    return {
        "exp_V_plus": ev_plus,
        "exp_V_minus": ev_minus,
        "susceptibility": chi,
    }

In [37]:
def get_int_counts_from_sampler_v2_result(pub_result):
    return pub_result.data.meas.get_int_counts()

def estimate_diagonal_cost_moments_from_counts(
    int_counts: dict[int, int],
    qubo_simple,
):
    """
    Uses your existing calculate_cost(state, qubo_simple).
    """
    shots = sum(int_counts.values())
    if shots == 0:
        return {
            "exp_cost": np.nan,
            "exp_cost_sq": np.nan,
            "var_cost": np.nan,
        }

    exp1 = 0.0
    exp2 = 0.0

    for state, ct in int_counts.items():
        p = ct / shots
        c = float(calculate_cost(state, qubo_simple))
        exp1 += p * c
        exp2 += p * (c ** 2)

    return {
        "exp_cost": exp1,
        "exp_cost_sq": exp2,
        "var_cost": exp2 - exp1 ** 2,
    }

In [38]:
def sample_prefix_cost_moments(
    sampler,
    pm,
    prefix_circuit,
    qubo_simple,
    shots: int = 10000,
):
    qc = prefix_circuit.copy()
    qc.measure_all()
    qc_t = pm.run(qc)

    job = sampler.run([(qc_t,)], shots=shots)
    result = job.result()[0]
    int_counts = get_int_counts_from_sampler_v2_result(result)

    moments = estimate_diagonal_cost_moments_from_counts(int_counts, qubo_simple)
    return moments, int_counts, qc_t

In [39]:
def entropy_from_counts(int_counts):

    shots = sum(int_counts.values())

    if shots == 0:
        return np.nan

    probs = np.array(list(int_counts.values()), dtype=float) / shots

    return -np.sum(probs * np.log(probs + 1e-12))

In [40]:
import pandas as pd

def run_gap_proxy_workflow(
    prefixes_by_alpha,
    estimator,
    sampler,
    pm,
    H_problem: SparsePauliOp,
    H_driver: SparsePauliOp,
    qubo_simple,
    num_steps: int,
    schedule_s,
    s_scale: float = 1.0,
    use_midpoint: bool = False,
    susceptibility_observable: SparsePauliOp | None = None,
    lam: float = 0.02,
    probe_dt: float = 0.05,
    precision: float = 0.02,
    shots: int = 10000,
    prefix_key: str = "compressed_prefix",
    csv_path: str | None = None,
):
    rows = []

    for alpha in sorted(prefixes_by_alpha.keys()):
        prefix = prefixes_by_alpha[alpha].get(prefix_key, None)
        if prefix is None:
            continue

        row = {"alpha": int(alpha)}


        # Estimator-based expectation values
        est_obs = measure_prefix_observables_with_estimator(
            estimator=estimator,
            prefix=prefix,
            pm=pm,
            H_problem=H_problem,
            H_driver=H_driver,
            precision=precision,
        )
        row.update(est_obs)

        # Sampler-based diagonal moments from counts
        samp_obs, int_counts, qc_t = sample_prefix_cost_moments(
            sampler=sampler,
            pm=pm,
            prefix_circuit=prefix,
            qubo_simple=qubo_simple,
            shots=shots,
        )
        row.update({
            "sample_exp_cost": samp_obs["exp_cost"],
            "sample_exp_cost_sq": samp_obs["exp_cost_sq"],
            "sample_var_cost": samp_obs["var_cost"],
            "prefix_twoq_depth": qc_t.depth(lambda x: x.operation.num_qubits == 2),
        })

        # Optional susceptibility proxy
        if susceptibility_observable is not None:
            H_frozen = frozen_hamiltonian_at_alpha(
                alpha=alpha,
                num_steps=num_steps,
                H_initial=H_driver,
                H_problem=H_problem,
                schedule_s=schedule_s,
                s_scale=s_scale,
                use_midpoint=use_midpoint,
            )

            chi_dict = estimate_susceptibility(
                estimator=estimator,
                prefix_circuit=prefix,
                pm=pm,
                H_frozen=H_frozen,
                V=susceptibility_observable,
                lam=lam,
                probe_dt=probe_dt,
                precision=precision,
            )
            row.update(chi_dict)

        entropy = entropy_from_counts(int_counts)
        row.update({"entropy": entropy})
        
        rows.append(row)

        
        if csv_path is not None:
            pd.DataFrame(rows).to_csv(csv_path, index=False)

    return pd.DataFrame(rows)

In [41]:
from qiskit_ibm_runtime import EstimatorV2 as Estimator
# from qiskit_ibm_runtime import SamplerV2 as Sampler

estimator = Estimator(mode=backend)
# sampler = Sampler(mode=backend)

In [70]:
V = H_initial

df_gap = run_gap_proxy_workflow(
    prefixes_by_alpha=prefixes,
    estimator=estimator,
    sampler=sampler,
    pm=pm,
    H_problem=H_problem,
    H_driver=H_initial,
    qubo_simple=qubo_simple,
    num_steps=num_steps,
    schedule_s=default_schedule_s,
    s_scale=1.0,
    use_midpoint=True,
    susceptibility_observable=V,
    lam=0.02,
    probe_dt=0.05,
    precision=0.02,
    shots=10000,
    prefix_key="compressed_prefix",
    csv_path="gap_proxy_results.csv",
)

print(df_gap)

   alpha  exp_H_problem  exp_H_problem_sq  var_H_problem  exp_H_driver  \
0      1      -0.139779          3.088517       3.068979     -1.363991   
1      2      -0.741221          2.488044       1.938636      0.056938   
2      3      -0.576431          2.590717       2.258445      0.433205   
3      4      -0.106078          3.259833       3.248580     -0.023958   
4      5      -0.216314          3.270173       3.223382     -0.008950   
5      6       0.222341          3.794900       3.745464     -0.153360   

   sample_exp_cost  sample_exp_cost_sq  sample_var_cost  prefix_twoq_depth  \
0        -1.236275            3.391412         1.863038                118   
1        -0.780523            3.243556         2.634341                125   
2        -1.257948            3.659225         2.076791                122   
3        -0.552364            4.009793         3.704688                252   
4        -0.605265            4.021532         3.655186                254   
5        -0.7

In [68]:
prefixes

{0: {'target_prefix': None, 'good_prefix': None, 'compressed_prefix': None},
 1: {'target_prefix': <qiskit.circuit.quantumcircuit.QuantumCircuit at 0x73361a4ecf90>,
  'good_prefix': <qiskit.circuit.quantumcircuit.QuantumCircuit at 0x73361a9a1090>,
  'compressed_prefix': <qiskit.circuit.quantumcircuit.QuantumCircuit at 0x733538184790>},
 2: {'target_prefix': <qiskit.circuit.quantumcircuit.QuantumCircuit at 0x7335381ccdd0>,
  'good_prefix': <qiskit.circuit.quantumcircuit.QuantumCircuit at 0x7335381cd810>,
  'compressed_prefix': <qiskit.circuit.quantumcircuit.QuantumCircuit at 0x7335357f5bd0>},
 3: {'target_prefix': <qiskit.circuit.quantumcircuit.QuantumCircuit at 0x7335357d9190>,
  'good_prefix': <qiskit.circuit.quantumcircuit.QuantumCircuit at 0x73353816f950>,
  'compressed_prefix': <qiskit.circuit.quantumcircuit.QuantumCircuit at 0x7335356ca6d0>},
 4: {'target_prefix': <qiskit.circuit.quantumcircuit.QuantumCircuit at 0x73353563df90>,
  'good_prefix': <qiskit.circuit.quantumcircuit.Quan

In [44]:
# for alpha in sorted(prefixes.keys()):
#     prefix_dict = prefixes[alpha]

#     target_prefix = prefix_dict.get("target_prefix", None)
#     if target_prefix is None:
#         continue

#     # transpile / compile
#     compiled = pm.run(target_prefix)

#     # two-qubit depth
#     twoq_depth = compiled.depth(
#         filter_function=lambda inst: inst.operation.num_qubits == 2
#     )

#     print(f"alpha={alpha}, two_qubit_depth={twoq_depth}")

In [45]:
# for alpha in sorted(prefixes.keys()):
#     prefix_dict = prefixes[alpha]

#     good_prefix = prefix_dict.get("good_prefix", None)
#     if target_prefix is None:
#         continue

#     # transpile / compile
#     compiled = pm.run(good_prefix)

#     # two-qubit depth
#     twoq_depth = compiled.depth(
#         filter_function=lambda inst: inst.operation.num_qubits == 2
#     )

#     print(f"alpha={alpha}, two_qubit_depth={twoq_depth}")

In [46]:
def run_qaoa_cvar_experiment(
    cost_operator,
    mixer,
    sampler,
    pm,
    vrp3,
    qubo_simple,
    num_variables,
    csv_path,
    qaoa_depths=(1, 2, 3, 4, 5, 6),
    shots=10000,
    maxiter=250,
    aggregation=0.2,   # CVaR alpha if float
):
    """
    For each QAOA depth p:
      - build plain QAOA ansatz
      - optimize with CVaR objective on QUBO
      - transpile final circuit
      - run final circuit with measurement
      - save paper metrics to CSV
    """
    rows = []

    for p in qaoa_depths:
        print(f"qaoa_p={p}")

        ansatz = QAOAAnsatz(cost_operator=cost_operator, reps=p,mixer_operator=mixer, flatten=True)
        ansatz.measure_all()

        # transpile parameterized ansatz to estimate actual 2q depth
        ansatz_for_depth = ansatz.copy()
        ansatz_t = pm.run(ansatz_for_depth)
        twoq_depth_for_cvar = ansatz_t.depth(
            lambda x: x.operation.num_qubits == 2
        )

        cvar_aggregation, lf, gamma_cx, gamma_circ = compute_cvar_aggregation_from_depth(
            backend=backend,
            n=ansatz.num_qubits,
            two_qubit_gate_depth=twoq_depth_for_cvar,
        )

        # optimize CVaR / mean QUBO cost
        res, history = optimize_prefixed_qaoa_cvar(
            ansatz=ansatz_t,
            sampler=sampler,
            qubo_simple=qubo_simple,
            aggregation=cvar_aggregation,
            maxiter=maxiter,
        )

        # bind final parameters
        param_list = list(ansatz_t.parameters)
        bind_dict = {param_list[i]: float(res.x[i]) for i in range(len(param_list))}
        qc_final = ansatz_t.assign_parameters(bind_dict, inplace=False)

        # final circuit 2q depth
        full_twoq_depth = qc_final.depth(
            lambda x: x.operation.num_qubits == 2
        )

        # run final circuit
        job = sampler.run([(qc_final,)], shots=int(shots))
        sampler_result = job.result()
        int_counts = sampler_result[0].data.meas.get_int_counts()

        # final metrics
        metrics = analyze_counts_int(
            int_counts=int_counts,
            vrp3=vrp3,
            num_variables=num_variables,
            shots=shots,
            qubo_simple=qubo_simple,
        )

        row = {
            "qaoa_p": int(p),
            "aggregation": aggregation,
            "num_iterations": getattr(res, "nfev", np.nan),
            "optimizer_success": getattr(res, "success", False),
            "final_objective": float(res.fun),
            "full_twoq_depth": full_twoq_depth,
            **metrics,
        }

        rows.append(row)
        pd.DataFrame(rows).to_csv(csv_path, index=False)

    return pd.DataFrame(rows)

In [47]:
# df_0qaoa = run_qaoa_cvar_experiment(
#     cost_operator=ising_H,
#     mixer=None,
#     sampler=sampler,
#     pm=pm,
#     vrp3=vrp3,
#     qubo_simple=qubo_simple,
#     num_variables=len(vrp3.variables),
#     csv_path="rensselaer_noPrefix_qaoa_cvar_results.csv",
#     qaoa_depths=p_list,
#     shots=10000,
#     maxiter=250,
#     aggregation=1,   # CVaR alpha
# )

In [48]:
# df_0qaoa_LC = run_qaoa_cvar_experiment(
#     cost_operator=ising_H_LC,
#     mixer=None,
#     sampler=sampler,
#     pm=pm,
#     vrp3=vrp3,
#     qubo_simple=qubo_simple,
#     num_variables=len(vrp3.variables),
#     csv_path="LC_rensselaer_noPrefix_qaoa_cvar_results.csv",
#     qaoa_depths=p_list,
#     shots=10000,
#     maxiter=250,
#     aggregation=1,   # CVaR alpha
# )

# XY mixer

In [64]:
df_qaoa_xy = run_prefixed_qaoa_cvar_experiment(
    prefixes_by_alpha=prefixes,
    cost_operator=ising_H,
    mixer="xy",
    sampler=sampler,
    pm=pm,
    vrp3=vrp3,
    qubo_simple=qubo_simple,
    num_variables=len(vrp3.variables),
    csv_path="XY_rensselaer_qaoa_cvar_results.csv",
    qaoa_depths=p_list,
    shots=10000,
    maxiter=250,
    aggregation=2,   # CVaR alpha
)

alpha=5, qaoa_p=1
Skipping p =  1
alpha=5, qaoa_p=2
Skipping p =  2
alpha=5, qaoa_p=3
Skipping p =  3
alpha=5, qaoa_p=4
Skipping p =  4
alpha=5, qaoa_p=5
Skipping p =  5
alpha=5, qaoa_p=6
layer fidelity 0.8411603597575423
alpha=6, qaoa_p=1
layer fidelity 0.8411603597575423
alpha=6, qaoa_p=2
layer fidelity 0.8411603597575423
alpha=6, qaoa_p=3
layer fidelity 0.8411603597575423
alpha=6, qaoa_p=4
layer fidelity 0.8411603597575423
alpha=6, qaoa_p=5
layer fidelity 0.8411603597575423
alpha=6, qaoa_p=6
layer fidelity 0.8411603597575423


In [67]:
df_LCqaoa_xy = run_prefixed_qaoa_cvar_experiment(
    prefixes_by_alpha=prefixes,
    cost_operator=ising_H_LC,
    mixer="xy",
    sampler=sampler,
    pm=pm,
    vrp3=vrp3,
    qubo_simple=qubo_simple,
    num_variables=len(vrp3.variables),
    csv_path="XYLC_rensselaer_qaoa_cvar_results.csv",
    qaoa_depths=p_list,
    shots=10000,
    maxiter=250,
    aggregation=1,   # CVaR alpha
)

Skipping alpha=0, missing prefix 'compressed_prefix'
alpha=1, qaoa_p=1
layer fidelity 0.8411603597575423
alpha=1, qaoa_p=2
layer fidelity 0.8411603597575423
alpha=1, qaoa_p=3
layer fidelity 0.8411603597575423
alpha=1, qaoa_p=4
layer fidelity 0.8411603597575423
alpha=1, qaoa_p=5
layer fidelity 0.8411603597575423
alpha=1, qaoa_p=6
layer fidelity 0.8411603597575423
alpha=2, qaoa_p=1
layer fidelity 0.8411603597575423
alpha=2, qaoa_p=2
layer fidelity 0.8411603597575423
alpha=2, qaoa_p=3
layer fidelity 0.8411603597575423
alpha=2, qaoa_p=4
layer fidelity 0.8411603597575423
alpha=2, qaoa_p=5
layer fidelity 0.8411603597575423
alpha=2, qaoa_p=6
layer fidelity 0.8411603597575423
alpha=3, qaoa_p=1
layer fidelity 0.8411603597575423
alpha=3, qaoa_p=2
layer fidelity 0.8411603597575423
alpha=3, qaoa_p=3
layer fidelity 0.8411603597575423
alpha=3, qaoa_p=4
layer fidelity 0.8411603597575423
alpha=3, qaoa_p=5
layer fidelity 0.8411603597575423
alpha=3, qaoa_p=6
layer fidelity 0.8411603597575423
alpha=4, qa